In [ ]:
!pip install rdkit pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 45.9 MB/s eta 0:00:00


In [8]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, MACCSkeys
from rdkit.Chem import rdMolDescriptors
import numpy as np
import pandas as pd
import os
import requests

In [15]:
df = pd.read_csv("egfr_ic50_raw.csv")

print(df.head())

   action_type activity_comment  activity_id activity_properties  \
0          NaN              NaN        32260                  []   
1          NaN              NaN        32263                  []   
2          NaN              NaN        32265                  []   
3          NaN              NaN        32267                  []   
4          NaN              NaN        32270                  []   

  assay_chembl_id                                  assay_description  \
0    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
1    CHEMBL621151  Inhibition of autophosphorylation of human epi...   
2    CHEMBL615325  Inhibition of ligand-induced proliferation in ...   
3    CHEMBL674637  Inhibitory activity towards tyrosine phosphory...   
4    CHEMBL621151  Inhibition of autophosphorylation of human epi...   

  assay_type  assay_variant_accession  assay_variant_mutation bao_endpoint  \
0          B                      NaN                     NaN  BAO_0000190   
1 

In [17]:

def smiles_to_mol(smiles):
    """Parse SMILES, return None if invalid."""
    mol = Chem.MolFromSmiles(smiles)
    return mol  # RDKit returns None for invalid SMILES

def compute_lipinski_descriptors(mol):
    """Lipinski Rule of Five descriptors."""
    return {
        'MW':   Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'HBD':  rdMolDescriptors.CalcNumHBD(mol),
        'HBA':  rdMolDescriptors.CalcNumHBA(mol),
        'TPSA': Descriptors.TPSA(mol),
        'RotBonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'RingCount': rdMolDescriptors.CalcNumRings(mol),
        'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
    }

def compute_morgan_fp(mol, radius=2, n_bits=1024):
    """Morgan (ECFP4) fingerprint as numpy array."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def compute_maccs_fp(mol):
    """MACCS 166-bit structural keys."""
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)

# Apply to dataset
mols = df['canonical_smiles'].apply(smiles_to_mol)
valid_mask = mols.notna()
df = df[valid_mask].copy()
mols = mols[valid_mask]

# Descriptor DataFrame
desc_df = pd.DataFrame([compute_lipinski_descriptors(m) for m in mols])
desc_df.index = df.index

# Morgan fingerprints (1024 bits)
morgan_fp = np.array([compute_morgan_fp(m) for m in mols])

# MACCS keys (166 bits)
maccs_fp = np.array([compute_maccs_fp(m) for m in mols])

# Combined feature matrix: descriptors + Morgan FP
df = df.dropna(subset=['canonical_smiles', 'standard_value'])

# Keep positive IC50 values only
df = df[df['standard_value'].astype(float) > 0]

# Convert IC50 (nM) → pIC50
df['pIC50'] = -np.log10(df['standard_value'].astype(float) * 1e-9)

# Binary activity label
df['active'] = (df['pIC50'] >= 6).astype(int)

print(df[['pIC50', 'active']].head())

X_descriptors = desc_df.values
X_combined    = np.hstack([X_descriptors, morgan_fp])
y_pIC50       = df['pIC50'].values
y_active      = df['active'].values

print(f"Feature matrix shape: {X_combined.shape}")
print(f"Descriptor names: {list(desc_df.columns)}")

[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerator
[10:08:46] DEPRECATION WARNING: please use MorganGenerat

      pIC50  active
0  7.387216       1
1  6.522879       1
2  5.106793       0
3  6.769551       1
4  7.397940       1
Feature matrix shape: (1000, 1032)
Descriptor names: ['MW', 'LogP', 'HBD', 'HBA', 'TPSA', 'RotBonds', 'RingCount', 'AromaticRings']
